# UAB · Day 3: Build a fermentation simulator

**Student exercises.** Replace the TODOs, then run the check cell after each exercise.

Assemble the original nine-state yeast model, handle feed transitions, and quantify feedstock and parameter variability.

Deliver a reproducible fed-batch study with balances, sensitivity, and exported data.

Time is in hours unless specified; model concentrations are g/L and volume is L.

Source: YeastFermentation Days 1–3. Python translation: Johan Le Nepvou De Carfort. Original MATLAB teaching material: Pedram Ramin and Krist Gernaey, PROSYS, DTU.

Run the embedded support-file setup once. For local Jupyter install numpy, scipy, and matplotlib first.


In [ ]:
# These support files are embedded for portable, offline notebook use.
# Original files retain their source content; course_models.py documents adapters.
import json
from pathlib import Path
_support = json.loads("{\"course_models.py\":\"\\\"\\\"\\\"Teaching adapters. Original equations/parameters remain in course_source.\\n\\nTime: h; concentration: g/L; volume: L; feed: L/h.\\nThe event/phase handling here corrects integration across feed discontinuities.\\n\\\"\\\"\\\"\\nimport random\\nimport numpy as np\\nfrom scipy.integrate import solve_ivp\\nfrom course_source.DAY3.parameters import Parameters as YeastParameters\\nfrom course_source.DAY3.model import initial_values, feed_values\\nfrom course_source.DAY3 import model as source_model\\n\\nSTATE_NAMES = ['Glucose', 'Xylose', 'Furfural', 'Furfuryl alcohol',\\n               '5-HMF', 'Acetic acid', 'Ethanol', 'Biomass', 'Volume']\\n\\ndef yeast_rhs(t, y, par, feed, feeding=False):\\n    # At t=0 the original Feed function is zero: recover reaction terms exactly.\\n    dy = source_model.fedbatch(0, y, par, feed)\\n    F = 0.1 + 0.005*(t-2) if feeding else 0.0\\n    dy[:8] += F * (np.asarray(feed[:8]) - y[:8]) / y[8]\\n    dy[8] = F\\n    return dy\\n\\ndef simulate_yeast(par=None, initial=None, end=50., method='BDF',\\n                   rtol=1e-7, atol=1e-9, feed=None):\\n    \\\"\\\"\\\"Integrate batch, feeding, and full-vessel phases on separate intervals.\\n\\n    Volume capacity is 2 L. The feed polynomial can be integrated analytically;\\n    split at the exact filling time instead of stepping over a state switch.\\n    \\\"\\\"\\\"\\n    from types import SimpleNamespace\\n    par = YeastParameters() if par is None else par\\n    y0 = np.asarray(initial_values() if initial is None else initial, dtype=float)\\n    if y0.shape != (9,) or not np.all(np.isfinite(y0)) or np.any(y0 < 0) or not 0 < y0[8] <= 2:\\n        raise ValueError('Use nine finite nonnegative states and 0 < V0 <= 2 L.')\\n    if not np.isfinite(end) or end <= 0:\\n        raise ValueError('end must be a finite positive time in hours.')\\n    feed = feed_values(y0.tolist()) if feed is None else list(feed)\\n    fill = 2 + (-0.1 + np.sqrt(0.1**2 + 0.01*(2-y0[8])))/0.005\\n    edges = sorted(set([0., float(end)] + [v for v in [2., fill] if 0 < v < end]))\\n    times, states, nfev = [], [], 0\\n    for a, b in zip(edges[:-1], edges[1:]):\\n        feeding = a >= 2 and a < fill and y0[8] < 2-1e-10\\n        grid = np.linspace(a, b, max(3, int((b-a)*12)+1))\\n        sol = solve_ivp(yeast_rhs, (a,b), y0, args=(par,feed,feeding),\\n                        method=method, t_eval=grid, rtol=rtol, atol=atol)\\n        if not sol.success:\\n            raise RuntimeError(sol.message)\\n        start = 1 if times else 0\\n        times.extend(sol.t[start:])\\n        states.extend(sol.y[:, start:].T)\\n        y0 = sol.y[:, -1].copy()\\n        nfev += sol.nfev\\n    return SimpleNamespace(t=np.array(times), y=np.array(states).T,\\n                           success=True, nfev=nfev, fill_time=fill)\\n\\ndef variable_feedstock(seed):\\n    \\\"\\\"\\\"BDG cell 7 ranges, local RNG; preserves zero-width FA draw.\\\"\\\"\\\"\\n    rng = random.Random(seed)\\n    bounds = [(35,40),(18,28),(0.6,2),(0,0),(0.5,2),(2,3),(0.2,0.9)]\\n    return [rng.uniform(lo,hi) for lo,hi in bounds] + [5.,0.7]\\n\\ndef variable_growth_parameters(seed):\\n    \\\"\\\"\\\"BDG cell 9: six independent +2 to +5% changes (one-sided).\\\"\\\"\\\"\\n    rng = random.Random(seed)\\n    par = YeastParameters()\\n    for name, baseline in [('numaxG',1.7271),('numaxX',1.6222),\\n                           ('YPSg',0.42),('YPSx',0.24),('YXSg',0.115),('YXSx',0.162)]:\\n        setattr(par, name, baseline*(1+rng.uniform(0.02,0.05)))\\n    return par\\n\\nclass ChemostatParameters:\\n    \\\"\\\"\\\"Values from DAY2/Control_Exercise_2, cell 3; no substituted mu_max.\\\"\\\"\\\"\\n    Ks, Ko, Yxs = 0.1, 0.2e-3, 0.4\\n    A, B, Eg, Gd, R, T = 1e10, 3e90, 58., 550., 8.314e-3, 293.15\\n    kla, Sf, V, F = 250., 10., 2., 0.5\\n    def __init__(self):\\n        self.mux = self.A*np.exp(-self.Eg/(self.R*self.T))/(1+self.B*np.exp(-self.Gd/(self.R*self.T)))\\n        self.O_sat = 0.21*0.027*np.exp(1142/self.T)*32/1000\\n\\ndef chemostat(t, y, par, kla=None):\\n    X, S, O = y\\n    mu = par.mux*S/(par.Ks+S)*O/(par.Ko+O)\\n    D = par.F/par.V\\n    transfer = par.kla if kla is None else kla\\n    return [ (mu-D)*X, D*(par.Sf-S)-mu*X/par.Yxs,\\n             transfer*(par.O_sat-O)-D*O-(1/par.Yxs-1)*mu*X ]\\n\\ndef oxygen_setpoint(t):\\n    return 0.003 if t > 10 else (0.006 if t > 5 else 0.005)\\n\\ndef oxygen_control(end=20., Kc=-50000., tauI=0.02):\\n    \\\"\\\"\\\"Original unsaturated PI; split at setpoint steps; returns actual actuation.\\\"\\\"\\\"\\n    from types import SimpleNamespace\\n    par = ChemostatParameters()\\n    if end <= 0 or tauI <= 0:\\n        raise ValueError('end and tauI must be positive.')\\n    y0 = [3.,0.,0.,0.]\\n    times, states = [], []\\n    edges = sorted(set([0., float(end)] + [v for v in [5.,10.] if v < end]))\\n    for a,b in zip(edges[:-1],edges[1:]):\\n        sp = oxygen_setpoint((a+b)/2)\\n        def rhs(t,y):\\n            X,S,O,I = y\\n            u = par.kla + Kc*(O-sp) + I\\n            return [*chemostat(t,[X,S,O],par,u), Kc/tauI*(O-sp)]\\n        sol = solve_ivp(rhs,(a,b),y0,method='BDF',t_eval=np.linspace(a,b,151),\\n                        rtol=1e-7,atol=[1e-9,1e-9,1e-11,1e-7])\\n        if not sol.success:\\n            raise RuntimeError(sol.message)\\n        start=1 if times else 0\\n        times.extend(sol.t[start:]); states.extend(sol.y[:,start:].T)\\n        y0=sol.y[:,-1]\\n    t=np.array(times); y=np.array(states).T\\n    sp=np.array([oxygen_setpoint(v) for v in t])\\n    u=par.kla + Kc*(y[2]-sp) + y[3]\\n    return SimpleNamespace(t=t,y=y,sp=sp,kla=u,success=True)\\n\",\"course_source/DAY3/model.py\":\"import numpy as np\\r\\n\\r\\ndef initial_values():\\r\\n    ## LIST OF INITIAL CONDITIONS.\\r\\n    # Name     Value     Index     Units\\r\\n    Glu0  =  39.7        # 1       g/L\\r\\n    Xyl0  =  23.5        # 2       g/L\\r\\n    Fur0  =  0.56        # 3       g/L\\r\\n    FA0   =  0           # 4       g/L\\r\\n    HMF0  =  0.2         # 5       g/L\\r\\n    HAc0  =  3.05        # 6       g/L\\r\\n    # Ac0   =  0.0001;    # 7       g/L\\r\\n    EtOH0 =  0.62        # 7       g/L\\r\\n    X0    =  1.75        # 8       g/L\\r\\n    V0    = 0.7          # 9       g/L\\r\\n    \\r\\n    init = [Glu0, Xyl0, Fur0, FA0, HMF0, HAc0, EtOH0, X0, V0]\\r\\n    return init\\r\\n\\r\\ndef feed_values(ini):\\r\\n    feed = ini.copy()\\r\\n    feed[7] = 0             # Xf = 0 !  \\r\\n    return feed\\r\\n\\r\\ndef Feed(t, V):\\r\\n    if t<2 or V>=2:\\r\\n        F = 0\\r\\n    else:\\r\\n        F = 0.1+0.005*(t-2)\\r\\n    return F\\r\\n\\r\\ndef fedbatch(t,x,par, Cfeed):\\r\\n    #  The following 8 components are considered in the model: \\r\\n    #  0. Glucose\\r\\n    #  1. Xylose\\r\\n    #  2. Furfural\\r\\n    #  3. Furfuryl alcohol\\r\\n    #  4. 5-HMF\\r\\n    #  5. HAc\\r\\n    #  6. Ethanol\\r\\n    #  7. Biomass\\r\\n    #  8. Tank volume\\r\\n\\r\\n    #  According to the kinetic model, the following chemical equations describe\\r\\n    #  the system. \\r\\n\\r\\n    #  1. Glucose  -----> Ethanol + Biomass\\r\\n    #  2. Xylose   -----> Ethanol + Biomass\\r\\n    #  3. Furfural -----> Furfuryl alcohol \\r\\n    #  4. HMF      -----> Acetic acid\\r\\n    #  5. HAc     <-----> Ac- [optional - not used here]\\r\\n    #  6. HAc      -----> Biomass (maintenence) \\r\\n\\r\\n\\r\\n    ## 1. STOICHIOMETRIC MATRIX. \\r\\n    # The 8 components and 6 reactions are considered in this model. One could\\r\\n    # also expand the model with Ac-, but that is not implemented.\\r\\n\\r\\n    #                Glu  Xyl  Fur  FA           HMF  HAc           EtOH        X  \\r\\n    st = np.array([[-1,   0,   0,   0,           0,   0,            par.YPSg,   par.YXSg],      # Glucose uptake\\r\\n                   [ 0,  -1,   0,   0,           0,   0,            par.YPSx,   par.YXSx],      # Xylose uptake\\r\\n                   [ 0,   0,  -1,   par.Y_FA_Fur,0,   0,            0,          0       ],      # Furfural uptake\\r\\n                   [ 0,   0,   0,   0,          -1,   par.Y_HAc_HMF,0,          0       ],      # HMF uptake\\r\\n                   [ 0,   0,   0,   0,           0,  -1,            0,          0       ]])     # HAc uptake\\r\\n\\r\\n\\r\\n\\r\\n\\r\\n    ## 2. DEFINE INDIVIDUAL KINETICS AND INHIBITION TERMS. \\r\\n    # Ph stands for phenomena, and it includes all the phenomena considered in\\r\\n    # the model, including reaction, inhibition and equilibria. Each of the\\r\\n    # phenomena is implemented separately below, for clarity\\r\\n    # 1. GLUCOSE UPTAKE RATE.\\r\\n    # numaxG: max consumption rate of glucose           s^-1       [1]\\r\\n    # KSPG:   affinity constant glucose                 g/L        [1]\\r\\n    # KiPG:   inhibition constant glucose               g/L        [1]\\r\\n    Ph1 = par.numaxG * x[0] * x[7] / (par.KSPG + x[0] + (x[0]**2)/par.KiPG)\\r\\n    Ph1 = max(0,Ph1)\\r\\n    # 2. XYLOSE UPTAKE RATE.\\r\\n    # numaxX: max consumption rate of xylose            s^-1       [1]\\r\\n    # KSPX:   affinity constant xylose                  g/L        [1]\\r\\n    # KiPX:   inhibition constant xylose                g/L        [1]\\r\\n    Ph2 = par.numaxX * x[1] * x[7] / (par.KSPX + x[1] + (x[1]**2)/par.KiPX)\\r\\n    Ph2 = max(0,Ph2)\\r\\n\\r\\n    # 3. FURFURAL UPTAKE RATE.\\r\\n    # numaxFur: max uptake rate of furfural               s^-1       [2]\\r\\n    # KSFur:    affinity constant furfural                g/L        [2]\\r\\n    Ph3 = par.numaxFur * x[2] * x[7] / (par.KSFur + x[2])\\r\\n    Ph3 = max(0,Ph3)\\r\\n\\r\\n    # 4. FURFURAL CONVERSION INTO FURFURYL ALCOHOL.\\r\\n    # Y_FA_Fur:  yield coefficient FA/Fur                  gFA/gFur   [2]\\r\\n    Ph4 = Ph3 * par.Y_FA_Fur\\r\\n    Ph4 = max(0,Ph4)\\r\\n\\r\\n    # 5. FURFURYL ALCOHOL INHIBITS GLUCOSE UPTAKE RATE.\\r\\n    # KiFAg:    inhibition constant of FA on Glu uptake   g/L        [2]\\r\\n    Ph5 = 1 / (1 + x[3]/par.KiFAg)\\r\\n    Ph5 = max(0,Ph5)\\r\\n\\r\\n    # 6. FURFURYL ALCOHOL INHIBITS XYLOSE UPTAKE RATE.\\r\\n    # KiFAx:    inhibition constant of FA on Xyl uptake   g/L        [2]\\r\\n    Ph6 = 1 / (1 + x[3]/par.KiFAx)\\r\\n    Ph6 = max(0,Ph6)\\r\\n\\r\\n    # 7. FURFURAL INHIBITS GLUCOSE UPTAKE RATE.\\r\\n    # KiFurg:   inhibition constant of Fur on Glu uptake  g/L        [2]\\r\\n    Ph7 = 1 / (1 + x[2]/par.KiFurg)\\r\\n    Ph7 = max(0,Ph7)\\r\\n\\r\\n    # 8. FURFURAL INHIBITS XYLOSE UPTAKE RATE.\\r\\n    # KiFurx:   inhibition constant of Fur on Xyl uptake  g/L        [2]\\r\\n    Ph8 = 1 / (1 + x[2]/par.KiFurx)\\r\\n    Ph8 = max(0,Ph8)\\r\\n\\r\\n    # 9. FURFURAL INHIBITS HMF UPTAKE RATE.\\r\\n    # KiFurHMF: inhibition constant of Fur on HMF         g/L        [2]\\r\\n    Ph9 = 1 / (1 + x[2]/par.KiFurHMF)\\r\\n    Ph9 = max(0,Ph9)\\r\\n\\r\\n    # 10. HMF INHIBITS GLUCOSE UPTAKE RATE.\\r\\n    # KiHMFg:   inhibition constant of HMF on Glu         g/L        [2]\\r\\n    Ph10 = 1 / (1 + x[4]/par.KiHMFg)\\r\\n    Ph10 = max(0,Ph10)\\r\\n\\r\\n    # 11. HMF INHIBITS XYLOSE UPTAKE RATE.\\r\\n    # KiHMFx:   inhibition constant of HMF on Xyl         g/L        [2]\\r\\n    Ph11 = 1 / (1 + x[4]/par.KiHMFx)\\r\\n    Ph11 = max(0,Ph11)\\r\\n\\r\\n    # 12. HMF UPTAKE RATE. \\r\\n    # numaxHMF: max uptake rate of HMF                    s^-1       [2]\\r\\n    # KSHMF:    affinity constant HMF                     g/L        [2]\\r\\n    Ph12 = par.numaxHMF * x[7] * x[4] / (par.KSHMF + x[4])\\r\\n    Ph12 = max(0,Ph12)\\r\\n\\r\\n    #  13. pH INFLUENCES THE PAIR HAc/Ac.\\r\\n    # pH = 5.5;              % pH is controlled\\r\\n    # pKa = 4.75;            % pKa of acetic acid.\\r\\n    # Ph13 = x(7,1) / x(6,1) * 1 / (1 + 10^(-pH)/10^(-pKa));\\r\\n    # Ph13 = max(0,Ph13);\\r\\n\\r\\n\\r\\n    # 14. HAc UPTAKE RATE.\\r\\n    # numaxHAc: max uptake rate of HAc                    s^-1       [2]\\r\\n    # KSHAc:    affinity constant HAc                     g/L        [2]\\r\\n    Ph14 = x[7] * x[5] * par.numaxHAc / (par.KSHAc + x[5])\\r\\n    Ph14 = max(0,Ph14)\\r\\n\\r\\n\\r\\n    #15. HMF CONVERSION INTO HAc\\r\\n    #Y_HAc_HMF = p(21);     % yield coefficient HAc/HMF                 gHAc/gHMF  [2], [3]\\r\\n    #Ph15 = Y_HAc_HMF * Ph12;\\r\\n    #Ph15 = max(0,Ph15);\\r\\n\\r\\n    # 16. HAc INHIBITS GLUCOSE UPTAKE RATE.\\r\\n    # KiHAcg:   inhibition constant of HAc on Glu         g/L        [2]\\r\\n    Ph16 = 1 / (1 + x[5]/par.KiHAcg)\\r\\n    Ph16 = max(0,Ph16)\\r\\n\\r\\n    # 17. HAc INHIBITS XYLOSE UPTAKE RATE.\\r\\n    # KiHAcx:   inhibition constant of HAc on Xyl         g/L         [2]\\r\\n    Ph17 = 1 / (1 + x[5]/par.KiHAcx)\\r\\n    Ph17 = max(0,Ph17)\\r\\n\\r\\n    #  18. PRODUCTION OF ETHANOL FORM GLUCOSE AND XYLOSE.\\r\\n    # YPSg:     yield ethanol-glucose                     gEtOH/gGlu [1]\\r\\n    # YPSx:     yield ethanol-xylose                      gEtOH/gGlu [1]\\r\\n    # Ph18a = par.YPSg * Ph1;\\r\\n    # Ph18b = par.YPSx * Ph2;\\r\\n    # Ph18a = max(0,Ph18a);\\r\\n    # Ph18b = max(0,Ph18b);\\r\\n\\r\\n    # 19. ETHANOL INHIBITS THE UPTAKE OF GLUCOSE AND XYLOSE.\\r\\n    # PMPg:     inhibition constant for glucose           g/L        [1]\\r\\n    # gammaG:   exponent factor inhibition glucose        g/L        [1]\\r\\n    # PMPx:     inhibition constant for xylose            g/L        [1]\\r\\n    # gammaX:   exponent factor inhibition xylose         g/L        [1]\\r\\n    Ph19a = 1-(x[6]/par.PMPg)**par.gammaG\\r\\n    Ph19b = 1-(x[6]/par.PMPx)**par.gammaX\\r\\n    Ph19a = max(0,Ph19a)\\r\\n    Ph19b = max(0,Ph19b)\\r\\n\\r\\n    #  20. CELL GROWTH.\\r\\n    # mGlu:     maintenance constant from glucose         g/L        [1]\\r\\n    # mXyl:     maintenance constant from xylose          g/L        [1]\\r\\n    # YXSg:     yield X-Glu                               gX/gGlu    [1]\\r\\n    # YXSx:     yield X-Xyl                               gX/gXyl    [1]\\r\\n    # mumaxG:   max growth of X from Glu                  h-1        [1]\\r\\n    # mumaxX:   max frowth of X from Xyl                  h-1        [1]\\r\\n\\r\\n    # Ph20a = max(0,(Ph1 + mGlu*x(8,1))*YXSg);\\r\\n    # Ph20b = max(0,(Ph1 + mXyl*x(8,1))*YXSx);\\r\\n\\r\\n    # 21. CATABOLITE REPRESSION.\\r\\n    # KiGlu:    inhibition constant of Glu to Xyl         g/L        Unknown\\r\\n    Ph21 = 1 / (1 + x[0]/par.KiGlu)\\r\\n\\r\\n    # 22. ACETATE IS USED FOR MAINTENANCE.\\r\\n    # mHAc:     maintenance constant from HAc              g/L        Unknown\\r\\n    # YXSHAc:   yield acetate biomass                      g/L        Unknown\\r\\n    Ph22 = max(0,par.mHAc * x[7] * par.YXSHAc)\\r\\n\\r\\n\\r\\n\\r\\n    ## 3. REACTION RATES. \\r\\n    # This includes the reaction rates with inhibition terms.\\r\\n    rates = np.zeros(5)\\r\\n    rates[0]  = Ph1 * Ph5 * Ph7 * Ph10 * Ph16 * Ph19a                            # Glucose uptake rate considering inhibitions.\\r\\n    rates[1]  = Ph2 * Ph6 * Ph8 * Ph11 * Ph17 * Ph19b * Ph21                     # Xylose uptake rate considering inhibitions.\\r\\n    rates[2]  = Ph3                                                              # Furfural uptake rate.\\r\\n    rates[3]  = Ph12 *  Ph9                                                      # 5-HMF uptake rate considering inhibitions.\\r\\n    rates[4]  = Ph14\\r\\n    \\r\\n    ## 4. MASS BALANCE FOR THE SYSTEM.\\r\\n    # Cfeed = feed_values()\\r\\n    dxdt = np.zeros(9)\\r\\n    r, c = st.shape\\r\\n    for i in range(c):\\r\\n        dxdt[i] = sum([rates[j] * st[j,i] for j in range(r)]) + Feed(t, x[8])*(Cfeed[i]-x[i])/x[8]        # Components mass balance -> FedBatch\\r\\n    dxdt[8] =  Feed(t, x[8])                 # Tank volume mass balance\\r\\n    return dxdt\\r\\n\\r\\nif __name__=='__main__':\\r\\n    from parameters import Parameters\\r\\n    init = initial_values()\\r\\n    par = Parameters()\\r\\n    dxdt = fedbatch(0, init, par)\\r\\n    print(dxdt)\\r\\n\\r\\n\\r\\n\\r\\n\",\"course_source/DAY3/parameters.py\":\"# [1] Krishnan et al, 1999\\r\\n# [2] Hanly et al, 2004\\r\\n# [3] Mauricio-Iglesias et al, \\r\\n\\r\\nclass Parameters():\\r\\n    '''Define all Parameters to be used in the model in a class'''\\r\\n    def __init__(self):\\r\\n        ## 1. GLUCOSE UPTAKE RATE.\\r\\n        # numaxG = 2.6808  ;       # max consumption rate of glucose (1.0005)         h^-1       [1]\\r\\n        self.numaxG = 1.7271          # max consumption rate of glucose (1.0005)  h^-1       [1]\\r\\n        # self.KSPG   = 0.14637         # affinity constant glucose to ethanol      g/L        [1]\\r\\n        self.KSPG   = 0.565         # affinity constant glucose to ethanol      g/L        [1]\\r\\n        # self.KiPG   = 4752.8          # inhibition constant glucose to ethanol    g/L        [1]\\r\\n        self.KiPG   = 4890          # inhibition constant glucose to ethanol    g/L        [1]\\r\\n        # p(1:3) = [numaxG, KSPG, KiPG];\\r\\n\\r\\n        ## 2. XYLOSE UPTAKE RATE.\\r\\n        #numaxX = 4.4851;         # max consumption rate of xylose            h^-1       [1]\\r\\n        # self.numaxX = 7.0531          # max consumption rate of xylose            h^-1       [1]\\r\\n        self.numaxX = 1.6222          # max consumption rate of xylose            h^-1       [1]\\r\\n        # self.KSPX   = 0.58763         # affinity constant xylose to ethanol       g/L        [1]\\r\\n        self.KSPX   = 3.4         # affinity constant xylose to ethanol       g/L        [1]\\r\\n        # self.KiPX   = 21.608          # inhibition constant xylose to ethanol     g/L        [1]\\r\\n        self.KiPX   = 18.1          # inhibition constant xylose to ethanol     g/L        [1]\\r\\n        # p(4:6) = [numaxX, KSPX, KiPX];\\r\\n\\r\\n        ## 3. FURFURAL UPTAKE RATE.\\r\\n        # self.numaxFur = 0.14309       # max uptake rate of furfural               h^-1       [2]\\r\\n        self.numaxFur = 4.67e-5*3600       # max uptake rate of furfural               h^-1       [2]\\r\\n        self.KSFur    = 0.05          # affinity constant furfural                g/L        [2]\\r\\n        #p(7:8) = [numaxFur, KSFur];\\r\\n\\r\\n        ## 4. FURFURAL CONVERSION INTO FURFURYL ALCOHOL.\\r\\n        self.Y_FA_Fur = 1.02          # yield coefficient FA/Fur                  gFA/gFur   [2]\\r\\n        #p(9) = Y_FA_Fur;\\r\\n        \\r\\n        ## 5. FURFURYL ALCOHOL INHIBITS GLUCOSE UPTAKE RATE.\\r\\n        self.KiFAg = 5                # inhibition constant of FA on Glu uptake   g/L        [2]3\\r\\n        #p(10) = KiFAg;\\r\\n\\r\\n        ## 6. FURFURYL ALCOHOL INHIBITS XYLOSE UPTAKE RATE.\\r\\n        self.KiFAx = 6                # inhibition constant of FA on Xyl uptake   g/L        [2]\\r\\n        #p(11) = KiFAx;\\r\\n\\r\\n        ## 7. FURFURAL INHIBITS GLUCOSE UPTAKE RATE.\\r\\n        # self.KiFurg = 0.4529          # inhibition constant of Fur on Glu uptake  g/L        [2]\\r\\n        self.KiFurg = 0.75            # inhibition constant of Fur on Glu uptake  g/L        [2]\\r\\n        #p(12) = KiFurg;\\r\\n\\r\\n        ## 8. FURFURAL INHIBITS XYLOSE UPTAKE RATE.\\r\\n        self.KiFurx = 0.35            # inhibition constant of Fur on Xyl uptake  g/L        [2]\\r\\n        #p(13) = KiFurx;\\r\\n\\r\\n        ## 9. FURFURAL INHIBITS HMF UPTAKE RATE.\\r\\n        self.KiFurHMF = 0.25          # inhibition constant of Fur on HMF         g/L        [2]\\r\\n        #p(14) = KiFurHMF;\\r\\n\\r\\n        ## 10. HMF INHIBITS GLUCOSE UPTAKE RATE.\\r\\n        self.KiHMFg = 2               # inhibition constant of HMF on Glu         g/L        [2]\\r\\n        #p(15) = KiHMFg;\\r\\n\\r\\n        ## 11. HMF INHIBITS XYLOSE UPTAKE RATE.\\r\\n        self.KiHMFx = 10              # inhibition constant of HMF on Xyl         g/L        [2]\\r\\n        #p(16) = KiHMFx;\\r\\n\\r\\n        ## 12. HMF UPTAKE RATE. \\r\\n        # self.numaxHMF = 0.3154        # max uptake rate of HMF                    h^-1       [2]\\r\\n        self.numaxHMF = 8.76E-5*3600        # max uptake rate of HMF                    h^-1       [2]\\r\\n        self.KSHMF = 0.5              # affinity constant HMF                     g/L        [2]\\r\\n        #p(17:18) = [numaxHMF, KSHMF];\\r\\n\\r\\n        ## 13. pH INFLUENCES THE PAIR HAc/Ac.\\r\\n        self.pH = 5.5                 # pH is controlled\\r\\n        self.pKa = 4.75               # pKa of acetic acid.\\r\\n\\r\\n        ## 14. HAc UPTAKE RATE.\\r\\n        self.numaxHAc = 1.23E-5*3600*0.001     # max uptake rate of HAc                    h^-1       [2]\\r\\n        self.KSHAc = 2.5                 # affinity constant HAc                     g/L        [2]\\r\\n        #p(19:20) = [numaxHAc, KSHAc];\\r\\n\\r\\n        ## 15. HMF CONVERSION INTO HAc\\r\\n        self.Y_HAc_HMF = 0.534        # yield coefficient HAc/HMF                 gHAc/gHMF  [2], [3]\\r\\n        #p(21) = Y_HAc_HMF;\\r\\n\\r\\n        ## 16. HAc INHIBITS GLUCOSE UPTAKE RATE.\\r\\n        # KiHAcg = 5.1827;         # inhibition constant of HAc on Glu         g/L        [2]\\r\\n        self.KiHAcg = 5.6703          # inhibition constant of HAc on Glu         g/L        [2]\\r\\n        # p(22) = KiHAcg;\\r\\n\\r\\n        ## 17. HAc INHIBITS XYLOSE UPTAKE RATE.\\r\\n        # KiHAcx = 0.73;           # inhibition constant of HAc on Xyl         g/L        [2]\\r\\n        self.KiHAcx = 3.8291          # inhibition constant of HAc on Xyl         g/L        [2]\\r\\n        # p(23) = KiHAcx;\\r\\n\\r\\n        ## 18. PRODUCTION OF ETHANOL FORM GLUCOSE AND XYLOSE.\\r\\n        # YPSg = 0.42;             # yield ethanol-glucose                     gEtOH/gGlu [1]\\r\\n        self.YPSg = 0.42              # yield ethanol-glucose                     gEtOH/gGlu [1]\\r\\n\\r\\n        self.YPSx = 0.24              # yield ethanol-xylose                      gEtOH/gGlu [1]\\r\\n        # p(24:25) = [YPSg, YPSx];\\r\\n\\r\\n        ## 19. ETHANOL INHIBITS THE UPTAKE OF GLUCOSE AND XYLOSE.\\r\\n        self.PMPg = 103             # inhibition constant for glucose           g/L        [1]\\r\\n        self.gammaG = 1.42            # exponent factor inhibition glucose        g/L        [1]\\r\\n        self.PMPx = 60.2           # inhibition constant for xylose            g/L        [1]\\r\\n        # PMPx = 20.9778;          # inhibition constant for xylose            g/L        [1]\\r\\n        self.gammaX =0.608          # exponent factor inhibition xylose         g/L        [1]\\r\\n        # gammaX = 1.7376;         # exponent factor inhibition xylose         g/L        [1]\\r\\n        # p(26:29) = [PMPg, gammaG, PMPx, gammaX];\\r\\n\\r\\n        ## 20. CELL GROWTH.\\r\\n        self.mGlu = 2.69E-5           # maintenance constant from glucose         g/L        [1]\\r\\n        self.mXyl = 1.86E-5           # maintenance constant from xylose          g/L        [1]\\r\\n        self.YXSg = 0.115            # yield X-Glu                               gX/gGlu    [1]\\r\\n        self.YXSx = 0.162             # yield X-Xyl                               gX/gXyl    [1]\\r\\n        self.mumaxG = 0.3308          # max growth of X from Glu                  h-1        [1]\\r\\n        self.mumaxX = 1.0008          # max frowth of X from Xyl                  h-1        [1]\\r\\n        # KSPGgr = 0.565;         # affinity constant glucose to growth       g/L        [1]\\r\\n        # KiPGgr = 283.7;         # inhibition constant glucose to growth     g/L        [1]\\r\\n        # KSPXgr = 3.4;           # affinity constant xylose to growth        g/L        [1]\\r\\n        # KiPXgr = 18.1;          # inhibition constant xylose to growth      g/L        [1]\\r\\n        # PMXg   = 95.4 ;         # max EtOH concentration for growth glucose g/L        [1]\\r\\n        # PMXx   = 59.04;         # max EtOH concentration for growth xylose  g/L        [1]\\r\\n        # p(30:35) = [mGlu, mXyl,YXSg, YXSx, mumaxG, mumaxX];\\r\\n\\r\\n        ## 21. CATABOLITE REPRESSION.\\r\\n        self.KiGlu = 13.763                # inhibition constant of Glu to Xyl         g/L        Unknown\\r\\n        # p(36) = KiGlu;\\r\\n\\r\\n        ## 22. ACETIC ACID IS USED FOR MAINTENANCE.\\r\\n        self.mHAc = 0                 # maintenance constant from HAc              g/L        Unknown\\r\\n        self.YXSHAc = 0               # yield acetate biomass                      g/L        Unknown \\r\\n        # p(38:39) = [mHAc, YXSHAc];\\r\\n\\r\\n\\r\\n        ## COLLECT PARAMETER NAMES\\r\\n        self.par_names = {'numaxG', 'KSPG', 'KiPG', 'numaxX', 'KSPX', 'KiPX', 'numaxFur', 'KSFur', 'Y_FA_Fur', 'KiFAg', 'KiFAx', 'KiFurg',\\r\\n            'KiFurx', 'KiFurHMF', 'KiHMFg', 'KiHMFx', 'numaxHMF', 'KSHMF', 'numaxHAc', 'KSHAc', 'Y_HAC_HMF', 'KiHAcg', 'KiHAcx', \\r\\n            'YPSg', 'YPSx', 'PMPg', 'gammaG', 'PMPx', 'gammaX', 'mGlu', 'mXyl', 'YXSg', 'YXSx', 'KiGlu'}\\r\\n\\r\\n        ## COLLECT DEPENDENT VARIABLE NAMES\\r\\n        self.var_names = ['Glucose', 'Xylose', 'Furfural','Furfuryl alcohol','5-HMF', 'Acetic acid', 'Ethanol', 'Biomass']\\r\\n\\r\\nif __name__=='__main__':\\r\\n    par = Parameters()\\r\\n    print(par.mHAc)\"}")
for _name, _content in _support.items():
    _path = Path(_name)
    _path.parent.mkdir(parents=True, exist_ok=True)
    _path.write_text(_content, encoding='utf-8')
print('Model support ready. Requires numpy, scipy, matplotlib.')


## 01. Read the yeast model before running it

**Objective:** Identify the nine states, five reaction rates, and source parameter values in the fed-batch model.

<p>The original DAY3 model contains eight concentration states followed by volume. Keep this order everywhere: glucose, xylose, furfural, furfuryl alcohol, 5-HMF, acetic acid, ethanol, biomass, volume. Initial concentrations are <code>[39.7,23.5,0.56,0,0.2,3.05,0.62,1.75]</code> g/L; volume is 0.7 L.</p><div class="equation">r<sub>reaction</sub> = Nᵀv &nbsp; · &nbsp; N shape = (5, 8)</div><p>Rows of N correspond to glucose uptake, xylose uptake, furfural conversion, HMF conversion, and acetic-acid uptake. For glucose uptake, the row is <code>[-1,0,0,0,0,0,YPSg,YXSg]</code>. DAY3 uses <code>YPSg=0.42</code> and <code>YXSg=0.115</code> g/g.</p><p>The sugar reactions produce both ethanol and biomass. This reduced matrix omits CO₂ and other products, so the sum of the eight concentrations is <strong>not</strong> a conserved mass. The model does not explicitly close carbon or elemental balances.</p>

**Task:** Construct the 5×8 stoichiometric matrix and calculate reaction derivatives for the supplied uptake-rate vector. Fill in the glucose product yields from the DAY3 parameter object.

Source: DAY3/BDG.ipynb, original zero-based cells 1, 2, 3, 4.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

from course_models import YeastParameters,initial_values,STATE_NAMES
par=YeastParameters()
N=np.array([
    [-1,0,0,0,0,0,0,0],  # TODO: glucose -> ethanol + biomass
    [0,-1,0,0,0,0,par.YPSx,par.YXSx],
    [0,0,-1,par.Y_FA_Fur,0,0,0,0],
    [0,0,0,0,-1,par.Y_HAc_HMF,0,0],
    [0,0,0,0,0,-1,0,0]],dtype=float)
rates=np.array([1.,0.5,0.01,0.02,0.001])  # g/L/h
reaction=N.T@rates
print("State order:",STATE_NAMES)
print("Initial states:",initial_values())
print("Reaction derivatives [g/L/h]:",reaction)


### Check your result
Run this after completing the exercise. An AssertionError indicates a result to investigate.


In [ ]:
assert N.shape==(5,8)
assert np.isclose(reaction[6],0.54), 'Ethanol comes from both sugars.'
assert np.isclose(reaction[7],0.196), 'Include both biomass yields.'
assert np.isclose(reaction[5],0.00968), 'Acetate is formed from HMF and consumed.'
print("All numerical checks passed.")


**Interpretation:** The source comments mention six reactions, but the implemented matrix has five rows. Teach and inspect the implemented equations; unused comments and parameters do not add reaction pathways.

**Hand in:** A state/units table and an annotated stoichiometric matrix. Explain why summing these concentrations is not a valid conservation test.


## 02. From uptake kinetics to inhibited reaction rates

**Objective:** Implement the original glucose uptake law and separate base uptake from inhibition factors.

<p>The glucose uptake law combines substrate saturation and substrate inhibition with biomass concentration. Its coefficient is <code>numaxG=1.7271</code> h⁻¹, <code>KSPG=0.565</code> g/L, and <code>KiPG=4890</code> g/L in DAY3. DAY1 has different fitted parameters; do not mix parameter sets.</p><div class="equation">vG,base = νmax,G · G · X / (Ks,G + G + G²/Ki,G)<br>vG = vG,base · IFA · IFur · IHMF · IHAc · IEtOH</div><p>Each soluble inhibitor contributes <code>1/(1+C/Ki)</code>. Ethanol contributes <code>max(0, 1-(P/PMPg)**gammaG)</code>. For xylose, an extra factor <code>1/(1+G/KiGlu)</code> accounts for glucose repression. The kinetic helper always uses the exact DAY3 source equations.</p><p>These are empirical kinetics. The product of inhibition factors can be much smaller than each individual factor. A high substrate concentration does not guarantee a high uptake rate.</p>

**Task:** Complete the base glucose uptake rate. Apply all five inhibitors and compare the result with the glucose derivative from the original model at t=0, when feed is off.

Source: DAY3/BDG.ipynb, original zero-based cells 1, 4, 9.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

from course_models import YeastParameters,initial_values,feed_values
from course_source.DAY3.model import fedbatch
par=YeastParameters()
y=np.array(initial_values())
G,_,Fur,FA,HMF,HAc,P,X,V=y
base=0.  # TODO: glucose uptake before soluble/product inhibition
inhibition=(1/(1+FA/par.KiFAg) * 1/(1+Fur/par.KiFurg)
            * 1/(1+HMF/par.KiHMFg) * 1/(1+HAc/par.KiHAcg)
            * max(0,1-(P/par.PMPg)**par.gammaG))
uptake=base*inhibition
reference=-fedbatch(0,y,par,feed_values(y.tolist()))[0]
print("Base:",base,"Inhibition:",inhibition,"Actual uptake:",uptake)
print("Source uptake [g/L/h]:",reference)


### Check your result
Run this after completing the exercise. An AssertionError indicates a result to investigate.


In [ ]:
assert base>0
assert 0<inhibition<=1
assert np.isclose(uptake,reference,rtol=1e-10), 'Use DAY3 parameters and all inhibition factors.'
assert uptake<base
print("All numerical checks passed.")


**Interpretation:** Clamping each uptake rate to nonnegative values is part of the original implementation. It does not guarantee that a poorly resolved numerical trajectory cannot step into negative concentrations.

**Hand in:** A factor-by-factor inhibition table. Set each inhibitor concentration to zero in turn and explain which change most affects initial glucose uptake.


## 03. Derive and solve the fed-batch balance

**Objective:** Derive dilution from an extensive balance and resolve feed-start and vessel-capacity transitions.

<p>Start with the amount balance <code>d(Ci*V)/dt = ri*V + F*Cif</code> and the volume balance <code>dV/dt=F</code>. Applying the product rule produces the concentration dilution term; adding feed concentration alone would be incorrect.</p><div class="equation">dCi/dt = ri + (F/V)(Cif − Ci)<br>dV/dt = F</div><p>The original feed is zero before 2 h, then <code>F=0.1+0.005*(t-2)</code> L/h until V reaches 2 L. After capacity, feed stays off. The feed composition equals the initial composition except that feed biomass is zero. It includes the original nonzero ethanol concentration.</p><p>During feeding, <code>V=V0+0.1τ+0.0025τ²</code> with τ=t−2. For V0=0.7 L, capacity is reached at approximately 12.3315 h. The adapter splits integration at 2 h and this analytically calculated filling time so an adaptive step cannot overshoot the volume switch.</p>

**Task:** Run the complete model and reconstruct the analytical volume trajectory. Replace the placeholder volume expression with the integrated feed law, clipped at the 2 L capacity. Plot glucose, xylose, ethanol, biomass, and volume.

Source: DAY3/BDG.ipynb, original zero-based cells 1, 3, 4.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

from course_models import simulate_yeast
sol=simulate_yeast()
tau=np.maximum(sol.t-2,0)
expected_volume=np.full_like(sol.t,0.7)  # TODO: integrate feed, then limit to 2 L
print("Fill time [h]:",sol.fill_time)
print("Final ethanol [g/L]:",sol.y[6,-1])
fig,ax=plt.subplots(2,1,sharex=True)
for i,label in [(0,"Glucose"),(1,"Xylose"),(6,"Ethanol"),(7,"Biomass")]:
    ax[0].plot(sol.t,sol.y[i],label=label)
ax[0].set_ylabel("Concentration [g/L]"); ax[0].legend()
ax[1].plot(sol.t,sol.y[8],label="Numerical"); ax[1].plot(sol.t,expected_volume,"--",label="Balance")
ax[1].set(xlabel="Time [h]",ylabel="Volume [L]"); ax[1].legend(); fig.tight_layout()


### Check your result
Run this after completing the exercise. An AssertionError indicates a result to investigate.


In [ ]:
assert sol.success
assert np.max(abs(sol.y[8]-expected_volume))<1e-6, 'Integrate F and stop at vessel capacity.'
assert np.max(sol.y[8])<=2+1e-6
assert np.min(sol.y[:8])>-1e-6
assert np.isclose(sol.fill_time,12.3315017762,atol=1e-7)
print("All numerical checks passed.")


**Interpretation:** Concentration and amount can move in opposite directions during feeding. Biomass concentration is diluted by sterile feed even when biomass amount increases. Time is in hours; the original notebook incorrectly labels some plots in seconds.

**Hand in:** A product-rule derivation, a volume-balance check, and labelled concentration/volume plots.


## 04. Reproducible variability across fermentation runs

**Objective:** Separate feedstock variation from kinetic variation and reproduce the original seeded experiment.

<p>The original generator draws glucose from 35–40, xylose from 18–28, furfural from 0.6–2, HMF from 0.5–2, acetic acid from 2–3, and ethanol from 0.2–0.9 g/L. Initial biomass is 5 g/L and volume is 0.7 L. Furfuryl alcohol is zero. The helper preserves the original random draw order.</p><p>Six kinetic parameters are each multiplied by <code>1 + uniform(0.02,0.05)</code>: the two maximum sugar uptake coefficients, the two ethanol yields, and the two biomass yields. These changes are <strong>all positive</strong>, not ±5% noise.</p><div class="equation">pᵢ,run = pᵢ,baseline (1 + δᵢ) &nbsp; · &nbsp; δᵢ ∈ [0.02, 0.05]</div><p>The notebook reseeds feedstock and growth draws separately with the same run index. This makes results reproducible but couples the streams. The adapters reproduce this design with local random generators and fresh parameter objects. For uncertainty estimation, you would usually design independent streams and justify the parameter distributions.</p>

**Task:** Run the ten original seeds, applying both feedstock and growth variability. Replace the placeholder initial state with variable_feedstock(seed). Report final ethanol for each run and overlay its trajectories.

Source: DAY3/BDG.ipynb, original zero-based cells 5, 6, 7, 8, 9, 10, 11.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

from course_models import simulate_yeast,variable_feedstock,variable_growth_parameters,initial_values
final=[]
for seed in range(10):
    initial=initial_values()  # TODO: use the seeded feedstock generator
    par=variable_growth_parameters(seed)
    sol=simulate_yeast(par=par,initial=initial)
    final.append(sol.y[6,-1])
    plt.plot(sol.t,sol.y[6],alpha=0.6)
final=np.array(final)
print("Final ethanol [g/L]:",final)
print("Mean, sample SD:",final.mean(),final.std(ddof=1))
plt.xlabel("Time [h]"); plt.ylabel("Ethanol [g/L]")


### Check your result
Run this after completing the exercise. An AssertionError indicates a result to investigate.


In [ ]:
assert len(final)==10 and np.all(np.isfinite(final))
assert np.allclose(initial,variable_feedstock(9)), 'Apply the variable feedstock, not the baseline initial state.'
assert variable_feedstock(3)==variable_feedstock(3), 'The same seed must reproduce the same feedstock.'
assert final.std(ddof=1)>0
print("All numerical checks passed.")


**Interpretation:** A ten-run spread is a simulation study under chosen assumptions. It is not a calibrated confidence interval for a real fermenter. Compare feedstock-only, growth-only, and combined variation to interpret its causes.

**Hand in:** A reproducible run table with seeds and parameters; compare sources of variability using three separate experiments.


## 05. Sensitivity, amount balances & data export

**Objective:** Quantify a parameter sensitivity without confounding it with random changes, and export simulation data.

<p>A controlled sensitivity study changes one parameter while holding all other values fixed. Use terminal ethanol amount <code>M=P*V</code> rather than concentration alone. A centred finite difference estimates a dimensionless local sensitivity.</p><div class="equation">Sₚ ≈ [M(p(1+δ)) − M(p(1−δ))] / (2δM(p))</div><p>Use δ=0.01, then repeat at 0.005 to check finite-difference stability. This sensitivity includes the effects of the entire nonlinear model, including ethanol inhibition. It need not equal the sensitivity suggested by a single yield term.</p><p>Export time, the eight concentration states, and volume with explicit column units. Generated files live in the browser's Python filesystem until you download them. The course runner exposes CSV and PNG files created in the current exercise.</p>

**Task:** Calculate the centred normalized sensitivity of terminal ethanol amount to YPSg. Replace the placeholder terminal metric with P*V. Export the baseline trajectory to fermentation.csv.

Source: DAY3/BDG.ipynb, original zero-based cells 3, 4, 9, 11.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

from course_models import YeastParameters,simulate_yeast
delta=0.01
metrics=[]
for scale in [1-delta,1.,1+delta]:
    par=YeastParameters()
    par.YPSg*=scale
    sol=simulate_yeast(par=par)
    metric=sol.y[6,-1]  # TODO: terminal ethanol AMOUNT in grams
    metrics.append(metric)
    if scale==1.:
        baseline=sol
sensitivity=(metrics[2]-metrics[0])/(2*delta*metrics[1])
header="time_h,glucose_g_L,xylose_g_L,furfural_g_L,FA_g_L,HMF_g_L,HAc_g_L,ethanol_g_L,biomass_g_L,volume_L"
np.savetxt("fermentation.csv",np.column_stack([baseline.t,baseline.y.T]),delimiter=",",header=header,comments="")
print("Terminal ethanol amounts [g]:",metrics)
print("Normalized YPSg sensitivity:",sensitivity)


### Check your result
Run this after completing the exercise. An AssertionError indicates a result to investigate.


In [ ]:
assert np.isclose(metrics[1],baseline.y[6,-1]*baseline.y[8,-1]), 'Amount equals concentration times volume.'
assert np.isfinite(sensitivity)
data=np.loadtxt('fermentation.csv',delimiter=',',skiprows=1)
assert data.shape[1]==10 and np.allclose(data[:,0],baseline.t), 'Export time and all nine states.'
print("All numerical checks passed.")


**Interpretation:** The normalization makes sensitivity unitless: an S near 0.5 means a 1% parameter increase produces approximately a 0.5% change in this output near the baseline.

**Hand in:** A sensitivity comparison for δ=0.01 and 0.005, a clearly defined output metric, and a CSV with units.


## 06. Final lab: defend your fermentation simulation

**Objective:** Deliver a model study with explicit assumptions, numerical verification, reproducibility, and engineering interpretation.

<p>Your final deliverable is a small technical report and executable notebook. Use the original DAY3 kinetics, initial states, and feed policy. Your numerical implementation should handle the two feed transitions explicitly.</p><div class="equation">d(CᵢV)/dt = rᵢV + FCᵢf</div><p>For a controlled balance test, turn off all reaction coefficients. Glucose amount must then equal its initial amount plus glucose added with feed: <code>G*V = G0*V0 + Gfeed*(V-V0)</code>. For sterile feed, biomass amount stays constant when reaction is off. These extensive balances are stronger tests than expecting the sum of modelled concentrations to be constant.</p><p>For the reacting model, compare BDF and Radau with tight tolerances on the same output grid. Report the largest statewise differences, investigate negativity beyond roundoff, and state which physical effects the model omits. Keep Days 4 and 5 outside the study.</p>

**Task:** Disable all five uptake coefficients, run the no-reaction model, and verify glucose and biomass amount balances. Then run the reacting baseline separately and prepare the report described below.

Source: DAY3/BDG.ipynb, original zero-based cells 1, 3, 4, 7, 9, 11.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

from course_models import YeastParameters,simulate_yeast,initial_values
par=YeastParameters()
# TODO: turn off numaxG, numaxX, numaxFur, numaxHMF, numaxHAc

ini=np.array(initial_values())
sol=simulate_yeast(par=par,initial=ini)
Gmass=sol.y[0]*sol.y[8]
Gexpected=ini[0]*ini[8]+ini[0]*(sol.y[8]-ini[8])
Xmass=sol.y[7]*sol.y[8]
glucose_residual=np.max(abs(Gmass-Gexpected))
biomass_residual=np.max(abs(Xmass-ini[7]*ini[8]))
print("No-reaction glucose amount residual [g]:",glucose_residual)
print("No-reaction biomass amount residual [g]:",biomass_residual)
plt.plot(sol.t,Xmass); plt.xlabel("Time [h]"); plt.ylabel("Biomass amount [g]")


### Check your result
Run this after completing the exercise. An AssertionError indicates a result to investigate.


In [ ]:
assert all(getattr(par,n)==0 for n in ['numaxG','numaxX','numaxFur','numaxHMF','numaxHAc']), 'Disable all five reaction-rate coefficients.'
assert glucose_residual<1e-5, 'Check the extensive glucose balance.'
assert biomass_residual<1e-5, 'With no growth and sterile feed, biomass amount is conserved.'
assert np.max(sol.y[8])<=2+1e-6
print("All numerical checks passed.")


**Interpretation:** A correct no-reaction balance isolates flow and dilution from kinetics. The baseline study then combines these verified balances with the original reaction model. Neither a solver success flag nor a smooth plot replaces this verification.

**Hand in:** Submit: (1) state table and units; (2) reaction and feed equations; (3) phase handling; (4) no-reaction balance residuals; (5) BDF/Radau comparison; (6) ten seeded runs; (7) one parameter sensitivity; (8) CSV and reproducible notebook; (9) model limitations. Assessment: formulation 25%, numerical verification 30%, reproducibility 20%, engineering interpretation 25%.
